In [5]:
# ============================================================
# MULTI-LLM REASONING WORKFLOW
#
# OpenAI  : Generates a challenging reasoning question.
# Claude  : Solves the reasoning question.
# Gemini  : Explains Claude's solution in detail.
#
# Configuration is loaded from the .env file.
# ============================================================


# ============================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================================

import os

from dotenv import load_dotenv

from openai import OpenAI
from anthropic import Anthropic
from google import genai

from IPython.display import Markdown, display


# ============================================================
# STEP 2: LOAD ENVIRONMENT VARIABLES
# ============================================================

# Load variables from the .env file.
# override=True ensures the .env values take precedence
# over previously loaded environment variables.

load_dotenv(override=True)
print("Environment variables loaded successfully!")


# ============================================================
# STEP 3: READ API KEYS AND MODEL CONFIGURATIONS
# ============================================================

# Read API keys.

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "").strip()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()


# Read model names.

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "").strip()
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "").strip()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "").strip()


# Read optional Anthropic endpoint.

ANTHROPIC_BASE_URL = os.getenv("ANTHROPIC_BASE_URL", "").strip()


# ============================================================
# STEP 4: VALIDATE CONFIGURATION
# ============================================================

required_variables = {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "ANTHROPIC_API_KEY": ANTHROPIC_API_KEY,
    "GEMINI_API_KEY": GEMINI_API_KEY,

    "OPENAI_MODEL": OPENAI_MODEL,
    "CLAUDE_MODEL": CLAUDE_MODEL,
    "GEMINI_MODEL": GEMINI_MODEL
}


# Identify missing or empty environment variables
missing_variables = [
    variable                                  # Store the variable name
    for variable, value in required_variables.items()  # Iterate through dictionary key-value pairs
    if not value                             # Select variables whose values are empty
]

# Check whether any missing variables were found
if missing_variables:

    # Raise an error and display the names of missing variables
    raise ValueError(
        "Missing configuration in .env file: "
        + ", ".join(missing_variables)  # Combine variable names into a comma-separated string
    )


print("\nAll API keys loaded successfully!")
print("All model configurations loaded successfully!")


# ============================================================
# STEP 5: DISPLAY MODEL CONFIGURATION
# ============================================================

print("\n========== MODEL CONFIGURATION ==========")
print("OpenAI Model :", OPENAI_MODEL)
print("Claude Model:", CLAUDE_MODEL)
print("Gemini Model:", GEMINI_MODEL)
print("Anthropic Endpoint:",
      ANTHROPIC_BASE_URL if ANTHROPIC_BASE_URL
      else "Official Anthropic API")
print("=========================================")


# ============================================================
# STEP 6: INITIALIZE AI CLIENTS
# ============================================================

# Initialize OpenAI client.

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)


# Initialize Claude client.
#
# If ANTHROPIC_BASE_URL is configured, use that endpoint.
# Otherwise, use the official Anthropic API endpoint.

anthropic_endpoint = (
    ANTHROPIC_BASE_URL
    if ANTHROPIC_BASE_URL
    else "https://api.anthropic.com"
)


claude_client = Anthropic(
    api_key=ANTHROPIC_API_KEY,
    base_url=anthropic_endpoint
)


# Initialize Gemini client.

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


print("\nAll AI clients initialized successfully!")


# ============================================================
# STEP 7: DEFINE OPENAI SYSTEM PROMPT
# ============================================================

# ICARE Prompt Framework:
#
# I - Identity
# C - Context
# A - Action
# R - Result
# E - Example
#
# OpenAI will generate one challenging reasoning question.
# It must not provide the answer.

openai_system_prompt = """
IDENTITY:

You are an expert AI evaluation specialist and reasoning
assessment designer.

CONTEXT:

You are responsible for creating challenging reasoning
questions that evaluate the analytical capabilities of
advanced AI models.

ACTION:

Create exactly ONE challenging reasoning question.

The question must:

1. Test critical thinking and analytical reasoning.
2. Require understanding of multiple interconnected factors.
3. Be non-mathematical.
4. Have sufficient context to be independently understood.
5. Have a clear and defensible answer.
6. Avoid ambiguous wording.
7. Require deeper reasoning rather than simple memorization.

RESULT:

Return only the question.
Do not provide the answer.
Do not provide hints.
Do not provide an explanation.
Do not generate multiple questions.

EXAMPLE:

An organization introduces a policy intended to improve
transparency. After implementation, employees become less
willing to report mistakes, even though the policy makes
reporting easier and protects employees.

What underlying factor could explain this unexpected outcome?

IMPORTANT:

Do not reuse the example question.
Create an entirely different question.
"""


# ============================================================
# STEP 8: GENERATE REASONING QUESTION USING OPENAI
# ============================================================

print("\nGenerating a challenging question using OpenAI...\n")


try:
    openai_response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": openai_system_prompt
            },
            {
                "role": "user",
                "content": (
                    "Generate one challenging reasoning "
                    "question following all the instructions."
                )
            }
        ]
    )

    question = openai_response.choices[0].message.content

    if not question:
        raise ValueError("OpenAI returned an empty question.")

    print("OpenAI successfully generated the question!")

    display(
        Markdown(
            "## 🧠 QUESTION GENERATED BY OPENAI\n\n"
            + question
        )
    )

except Exception as error:
    print("\nOpenAI generation failed.")
    print("Error:", error)
    raise


# ============================================================
# STEP 9: SOLVE QUESTION USING CLAUDE
# ============================================================

print("\nGenerating a solution using Claude...\n")

claude_system_prompt = """
IDENTITY:

You are an expert analytical reasoning specialist.

CONTEXT:

You will receive a challenging reasoning question
generated by another AI model.

ACTION:

Analyze the question carefully and provide a detailed,
logically structured solution.

Your solution must include:

1. Understanding the problem.
2. Identification of the important factors.
3. Step-by-step reasoning.
4. Examination of possible alternative explanations.
5. Identification of the most convincing explanation.
6. A clear final answer.

RESULT:

Provide a comprehensive and logically consistent answer.

Do not simply guess.
Explain why your conclusion is reasonable.

If the question permits multiple defensible answers,
acknowledge that and explain the assumptions behind
your preferred answer.
"""


try:
    claude_response = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=3000,
        system=claude_system_prompt,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ]
    )


    # Extract text from Claude response blocks.

    claude_solution = "\n".join(
        block.text
        for block in claude_response.content
        if block.type == "text"
    )

    if not claude_solution:
        raise ValueError("Claude returned an empty solution.")

    print("Claude successfully generated the solution!")

    display(
        Markdown(
            "## 🤖 CLAUDE'S SOLUTION\n\n"
            + claude_solution
        )
    )

except Exception as error:
    print("\nClaude generation failed.")
    print("Error type:", type(error).__name__)
    print("Error details:", error)
    raise RuntimeError(
        "Claude could not generate a solution. "
        "Check the Anthropic API endpoint, API key, "
        "model ID, and model access."
    ) from error


# ============================================================
# STEP 10: EXPLAIN CLAUDE'S SOLUTION USING GEMINI
# ============================================================

print("\nAsking Gemini to explain Claude's solution...\n")


gemini_system_prompt = """
IDENTITY:

You are an expert educator, technical trainer,
and AI reasoning specialist.

CONTEXT:

Another AI model has already solved a challenging
reasoning question.

You must explain that solution to students.

ACTION:

Provide a detailed educational explanation.
Your explanation must include:

1. Restatement of the original question.
2. Simple explanation of the problem.
3. Explanation of Claude's reasoning.
4. Step-by-step breakdown of the solution.
5. Important concepts students should understand.
6. Why the final answer makes sense.
7. A practical real-world example.
8. Key takeaways for students.

RESULT:

Make the explanation easy to understand.
Use simple language.
Use headings and bullet points.
Avoid unnecessary jargon.
Do not blindly accept Claude's answer.
If you identify a logical weakness, explain it.
"""


gemini_request = f"""
ORIGINAL QUESTION:

{question}


CLAUDE'S SOLUTION:

{claude_solution}


YOUR TASK:

Explain Claude's solution in detail.
Teach the underlying reasoning to students
in a simple and structured manner.
"""


try:

    gemini_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=(
            gemini_system_prompt
            + "\n\n"
            + gemini_request
        )
    )


    gemini_explanation = gemini_response.text

    if not gemini_explanation:
        raise ValueError("Gemini returned an empty explanation.")

    print("Gemini successfully generated the explanation!")

    display(
        Markdown(
            "## 📚 GEMINI'S DETAILED EXPLANATION\n\n"
            + gemini_explanation
        )
    )


except Exception as error:
    print("\nGemini generation failed.")
    print("Error type:", type(error).__name__)
    print("Error details:", error)
    raise


# ============================================================
# STEP 11: WORKFLOW COMPLETED
# ============================================================

print("\n" + "=" * 65)
print("MULTI-LLM REASONING WORKFLOW COMPLETED SUCCESSFULLY!")
print("=" * 65)
print("\nOpenAI  : Question generation completed.")
print("Claude  : Solution generation completed.")
print("Gemini  : Detailed explanation completed.")
print("\nAll three AI models have completed their tasks!")
print("=" * 65)

Environment variables loaded successfully!

All API keys loaded successfully!
All model configurations loaded successfully!

========== MODEL CONFIGURATION ==========
OpenAI Model : gpt-5.5
Claude Model: claude-opus-5-5
Gemini Model: gemini-2.5-flash
Anthropic Endpoint: https://api.anthropic.com

All AI clients initialized successfully!

Generating a challenging question using OpenAI...

OpenAI successfully generated the question!


## 🧠 QUESTION GENERATED BY OPENAI

A hospital replaces informal shift handoffs with a standardized digital checklist to reduce communication errors. Six months later, audits show that staff complete the checklist consistently, but preventable care delays have increased. Interviews reveal that nurses now rarely discuss unusual patient concerns unless those concerns fit a checklist field, and junior staff assume that anything not entered is unimportant.

What is the most defensible explanation for why the intervention worsened outcomes despite improving standardization?


Generating a solution using Claude...

Claude successfully generated the solution!


## 🤖 CLAUDE'S SOLUTION

# Why the Checklist Worsened Outcomes

## 1. Understanding the Problem

The puzzle looks contradictory:
- **Process metric improved.** Checklist completion is consistent.
- **Outcome metric worsened.** Preventable care delays increased.
- **Mechanism evidence from interviews.** Nurses discuss unusual concerns only if they fit a field, and junior staff treat anything not entered as unimportant.

So the question is how an intervention can succeed on its own terms but fail at its purpose. The answer depends on what a handoff is actually for.

## 2. Important Factors

| Factor | Why it matters |
|---|---|
| **What handoffs transmit** | Routine data (vitals, meds, allergies) *and* tacit, contextual, anomalous information ("she seems more confused than yesterday," "family says he never sleeps this much"). |
| **What checklists capture well** | Predictable, categorizable information. |
| **What checklists capture poorly** | Novel, ambiguous, "weak signal" concerns that do not fit predefined categories. |
| **What was audited** | Completion (compliance), not the quality of information transfer. |
| **Staff experience gradient** | Junior staff rely more heavily on explicit cues and are less able to sense what is missing. |
| **Type of harm observed** | Delays typically arise from late recognition of deterioration, which is exactly the kind of information that is often unusual and hard to categorize. |

## 3. Step-by-Step Reasoning

**Step 1: The informal handoff did two jobs.** It transmitted data, and it supported shared sensemaking. Conversation let the outgoing nurse voice hunches and let the incoming nurse ask questions. Its weakness was inconsistency with routine data. Its strength was flexibility with unusual data.

**Step 2: The checklist replaced the conversation rather than supplementing it.** Interviews show that discussion now happens only when a concern fits a field. The checklist defined the boundaries of what counts as relevant, which was not its intended role.

**Step 3: Absence of information became a signal.** Junior staff assume that whatever is not entered is unimportant. A checklist implicitly claims to be complete. Under the old system, silence was ambiguous and prompted questions. Under the new one, silence reads as "nothing to worry about." This is a false sense of completeness, akin to automation bias. It is worst among less experienced staff, who lack the clinical intuition to notice gaps.

**Step 4: The error profile shifted rather than shrank.** The checklist probably did reduce omissions of routine information. It increased omissions of non-routine information. The early signs of deterioration that drive preventable delays are disproportionately non-routine. The intervention therefore traded low-consequence errors for high-consequence ones.

**Step 5: Measurement reinforced the problem.** Audits rewarded completion, so staff optimized for filling fields. This is a Goodhart's-law dynamic: once completion became the target, it stopped being a good proxy for effective communication. Nothing in the system signaled that something important was being lost.

**Step 6: Redundancy was lost.** Safety in complex systems depends partly on overlapping, redundant channels. Eliminating free-form dialogue removed a backup channel that had caught what structured fields missed.

## 4. Alternative Explanations

| Alternative | Assessment |
|---|---|
| **Confounders** (staffing cuts, higher acuity, seasonal surges, new protocols) | Plausible and must be checked. However, they would not explain the specific behavioral changes in the interviews. They could add to the deterioration but do not replace the mechanism. |
| **Detection artifact** (better documentation surfaces delays that were previously unrecorded) | Worth ruling out, for example by checking whether delay definitions or reporting sources changed. This also fits poorly with interviews describing reduced discussion of concerns. |
| **Poor checklist design only** (missing fields) | Partly true. But no finite checklist can anticipate every unusual concern. The deeper issue is that the checklist was treated as exhaustive, not just that it had gaps. |
| **Time burden** (checklist consumes time, leaving less for care) | Possible contributor. Data entry might crowd out conversation, which is compatible with the main explanation. |
| **Transition or learning curve** | Less likely after six months, and it would not produce the specific "not entered = unimportant" belief. |
| **Regression to the mean** (intervention launched after an unusually good period) | Possible statistically. Again, it does not explain the observed behavior change. |

## 5. Most Convincing Explanation

The most defensible explanation is that **the checklist changed handoffs from a dialogic, sensemaking process into a compliance-oriented documentation task.** Standardization captured routine information reliably. But staff came to treat the checklist as the complete definition of relevant information. As a result:

1. **Unusual concerns were filtered out** because they had no designated field.
2. **Missing information was misread as reassurance**, especially by junior staff.
3. **Audits measured completion rather than understanding**, so they rewarded the behavior and hid the loss.

The net effect was a decline in the transfer of exactly the anomalous, early-warning information that prevents care delays.

**Caveat:** This conclusion is strongly supported by the interview evidence, which supplies a coherent mechanism. Strict causal attribution still requires ruling out confounders and reporting changes, for example by comparing units, checking acuity and staffing data, or reviewing individual delay cases for missed handoff signals.

## 6. Final Answer

The intervention worsened outcomes because **standardization was implemented as a substitute for clinical conversation rather than a scaffold for it.** The checklist became an implicit boundary on what was worth saying. This suppressed the transfer of non-routine, tacit warning signs, and it taught less experienced staff to read silence as safety. Because audits measured checklist completion rather than shared understanding, the system appeared to improve while its most safety-critical function eroded.

**Implications for redesign:**
- Keep the structured core, but add mandatory open-ended prompts such as "What worries you about this patient?" and "What should I watch for?" I-PASS includes situational awareness and contingency elements for this reason.
- Require verbal, face-to-face exchange with read-back or synthesis by the receiver.
- Train staff explicitly that the checklist is the minimum, not the full picture.
- Audit outcomes and handoff quality, not just completion.


Asking Gemini to explain Claude's solution...

Gemini successfully generated the explanation!


## 📚 GEMINI'S DETAILED EXPLANATION

As your expert educator, technical trainer, and AI reasoning specialist, I'm here to guide you through a fascinating problem that an AI model (Claude) has expertly solved. This isn't just about finding the right answer; it's about understanding *why* that answer is correct and what valuable lessons it holds for us.

Let's dive in!

---

### **Understanding Why Good Intentions Can Go Wrong: A Hospital Case Study**

Imagine a situation where a solution designed to make things better actually makes them worse. This is a common challenge in complex systems, and our problem today illustrates it perfectly. We'll explore how standardizing a process can unintentionally harm critical human elements of communication.

---

### **1. Restatement of the Original Question**

Here's the problem we're trying to solve:

**"A hospital replaces informal shift handoffs with a standardized digital checklist to reduce communication errors. Six months later, audits show that staff complete the checklist consistently, but preventable care delays have increased. Interviews reveal that nurses now rarely discuss unusual patient concerns unless those concerns fit a checklist field, and junior staff assume that anything not entered is unimportant.**

**What is the most defensible explanation for why the intervention worsened outcomes despite improving standardization?"**

---

### **2. Simple Explanation of the Problem**

At its heart, this problem presents a puzzling contradiction:

*   **The Goal:** The hospital wanted to improve patient safety by making sure important information was always shared during shift changes. They thought a checklist would help.
*   **The Action:** They introduced a digital checklist for nurses to use when handing off patients to the next shift.
*   **The "Good" Result (Process Metric):** Nurses consistently completed the checklist. Great! Everyone was following the new system.
*   **The "Bad" Result (Outcome Metric):** Despite the checklist being completed, *more* patients experienced preventable care delays. This is the opposite of what they wanted!
*   **The Clue (Interviews):** When asked, nurses said they only talked about patient issues if they fit a specific box on the checklist. Junior staff even assumed that if something wasn't on the checklist, it wasn't important.

**The big question is:** How could a system designed to improve communication and standardization actually lead to *worse* patient outcomes and *more* delays? It seems counterintuitive!

---

### **3. Explanation of Claude's Reasoning (The Core Idea)**

Claude's solution gets to the heart of the problem by identifying a critical shift in how communication happened. Its core argument is that **the checklist, instead of being a *tool to support* human conversation and judgment, became a *replacement for it*.**

Here's the breakdown of Claude's main point:

*   **Original Handoff:** The old, informal handoff, while inconsistent, allowed for rich, human-to-human discussion. Nurses could share not just routine facts, but also "gut feelings," subtle observations, and unusual concerns that didn't fit into neat categories. This helped nurses understand the *full picture* of a patient.
*   **New Checklist Handoff:** The digital checklist, while excellent for capturing routine, easy-to-categorize information, unintentionally *shut down* that vital human conversation. It made staff believe that *only* what was on the list mattered.
*   **The Unintended Consequence:** This led to crucial, non-routine "red flag" information (the kind that often prevents delays) being ignored, misunderstood, or never even discussed. Junior staff, in particular, lacked the experience to question the checklist's completeness, assuming "no entry" meant "no problem."

In short, Claude argues that the hospital prioritized "checking boxes" over "making sense together," leading to a dangerous reduction in critical information transfer.

---

### **4. Step-by-Step Breakdown of the Solution**

Let's walk through Claude's logical steps to understand how it reached this conclusion:

#### **Step 1: The Dual Role of the Old Handoff**
*   **Before the checklist:** Informal handoffs weren't just about transferring facts (like vital signs or medications). They also allowed nurses to *interpret* and *make sense* of information together. An outgoing nurse might say, "Mrs. Smith's vitals are stable, but she just seems a bit 'off' to me, keep an eye on her mood." This "off-ness" is hard to put on a form but crucial for patient safety. This was its strength. Its weakness was that routine facts could sometimes be forgotten.

#### **Step 2: The Checklist *Replaced* Conversation, Not Supplemented It**
*   **The Shift:** The hospital likely intended the checklist to *add structure* to the conversation. However, the interviews clearly show that staff started treating it as the *entire conversation*. If a concern didn't fit a specific field, it often wasn't discussed. The checklist became a *boundary* for what was considered relevant, rather than a guide.

#### **Step 3: Silence Became a False Signal of Safety**
*   **The Danger:** In the old system, if an outgoing nurse didn't mention something, the incoming nurse might ask, "Is there anything else I should know?" or "How is the patient generally?" The silence prompted questions.
*   **The New Problem:** With the checklist, junior staff (who have less experience and rely more on explicit instructions) started to believe: "If it's not on the checklist, it must not be important." This created a false sense of security, where the *absence* of information was mistakenly interpreted as reassurance.

#### **Step 4: The Error Profile Changed – From Minor to Major**
*   **Fixing one problem, creating another:** The checklist probably did a good job reducing errors related to routine information (e.g., forgetting to mention a patient's allergy). However, it increased errors related to *non-routine, subtle, early warning signs* of patient deterioration. These "weak signals" are exactly what lead to preventable delays and worse outcomes. The intervention traded easily fixed, low-impact errors for harder-to-spot, high-impact errors.

#### **Step 5: Measurement Drove the Wrong Behavior**
*   **The Problem with Audits:** The hospital audited "checklist completion." This meant nurses focused on ticking all the boxes, even if it meant sacrificing the quality of the actual information transfer or overlooking critical, non-standard concerns. The audit measured *compliance with the tool* rather than *effective communication and patient safety outcomes*. This is a classic example where measuring the wrong thing can lead to unintended negative consequences.

#### **Step 6: Loss of Redundancy**
*   **Backup Systems:** In complex and risky environments like hospitals, safety often comes from having multiple ways to catch problems (redundancy). The informal conversation acted as a "backup channel" that could catch things the structured system might miss. By replacing the conversation entirely, this vital backup was removed, making the system more brittle.

---

### **5. Important Concepts Students Should Understand**

This case study highlights several crucial concepts:

*   **Tacit Knowledge vs. Explicit Knowledge:**
    *   **Explicit knowledge** is information that can be easily written down, categorized, and shared (like a patient's medication list). Checklists are great for this.
    *   **Tacit knowledge** is harder to express, based on experience, intuition, and context ("the patient just doesn't look right," "I have a bad feeling about this"). Informal conversations are essential for sharing this. The problem here was that the checklist prioritized explicit knowledge at the expense of tacit knowledge.
*   **Automation Bias:** This is the tendency to overly rely on automated systems or tools, sometimes to the point of ignoring contradictory information or failing to perform independent checks. Junior staff assuming "if it's not on the checklist, it's unimportant" is a clear example.
*   **Goodhart's Law:** This principle states: "When a measure becomes a target, it ceases to be a good measure." In our case, "checklist completion" (a measure of compliance) became the *target* for nurses, making it a poor indicator of *actual effective communication* and leading to negative outcomes.
*   **Systemic Safety:** Safety in complex systems isn't just about preventing individual errors; it's about how the entire system functions, including its ability to adapt, sense problems, and learn. Eliminating rich human communication removed a key part of the system's ability to be safe.
*   **The "Boundary Object" Effect:** A tool or process can inadvertently define the boundaries of what is considered relevant or important. The checklist became a boundary object, limiting the scope of what nurses felt they *could* or *should* communicate.

---

### **6. Why the Final Answer Makes Sense**

Claude's final answer—that **the intervention worsened outcomes because standardization was implemented as a substitute for clinical conversation rather than a scaffold for it**—makes perfect sense because:

*   **It directly addresses the contradiction:** It explains how improving a *process metric* (checklist completion) could lead to a *worsening outcome* (more delays). The improvement was superficial; the underlying, critical communication process was damaged.
*   **It aligns with the evidence:** The interviews are key. Nurses explicitly stating they *only* discuss what fits a field, and junior staff assuming unlisted items are unimportant, directly supports the idea that the checklist became a barrier, not a help, to comprehensive communication.
*   **It identifies the mechanism of harm:** By suppressing non-routine information and creating a false sense of security, the system actively prevented the early detection of patient deterioration—precisely what leads to preventable delays.
*   **It's comprehensive:** It considers various factors from human behavior to measurement bias, painting a complete picture of the problem.

This explanation doesn't just say *what* happened, but *how* and *why* it happened, linking the observed behaviors directly to the negative outcomes.

---

### **7. A Practical Real-World Example**

Let's consider a different, simpler scenario:

**Scenario: A Car Mechanic's Checklist**

Imagine a car repair shop that introduces a very detailed digital checklist for mechanics to use during routine car inspections. It has fields for "tire pressure," "oil level," "brake pad thickness," etc. The shop manager audits that every mechanic completes the checklist perfectly for every car.

**The Problem:** Over time, the shop notices an increase in customer complaints about unusual noises or subtle performance issues that weren't addressed during routine inspections. When asked, mechanics say, "Well, it wasn't on the checklist for a routine inspection, so I didn't write it down or bring it up" or "I assumed if the computer didn't flag it after I filled out the checklist, it wasn't a problem."

**The Explanation (paralleling the hospital case):**

*   **Before:** Experienced mechanics might notice a slightly unusual engine sound or a faint smell during a routine check, even if it wasn't a standard checklist item. They'd mention it to the customer or a colleague as a "watch item." This was their "tacit knowledge" and "sensemaking."
*   **After:** The detailed checklist became the *entire scope* of the inspection. Mechanics focused on ticking boxes for *explicit* items. The "unusual sound" or "faint smell" (the "non-routine concerns") didn't fit a checklist field, so they were ignored or dismissed. The "absence" of a box for "unusual engine sound" was misinterpreted as "no unusual engine sound."
*   **Outcome:** Routine maintenance improved, but customer complaints about subtle, non-standard problems increased, leading to bigger, more expensive repairs later (like preventable care delays). The checklist became a substitute for the mechanic's full diagnostic judgment and conversation, rather than a tool to aid it.

---

### **8. Key Takeaways for Students**

Here are the most important lessons from this case study:

*   **Standardization is a Tool, Not a Panacea:** Checklists and standardized processes are incredibly valuable, but they have limitations. They excel at routine, predictable tasks but can struggle with novel, ambiguous, or highly contextual information.
*   **Don't Confuse Compliance with Effectiveness:** Just because people are completing a process (compliance) doesn't mean it's achieving its intended outcome (effectiveness). Always look at the ultimate goal, not just the intermediate steps.
*   **Beware of Unintended Behavioral Changes:** New systems and tools can profoundly change how people behave and communicate, often in ways that designers don't anticipate. Always consider the human element and potential side effects.
*   **Value Tacit Knowledge and Human Judgment:** Experience-based intuition and the ability to "read between the lines" (tacit knowledge) are crucial, especially in complex environments like healthcare. Systems should *support* this, not suppress it.
*   **The Power of Dialogue and Sensemaking:** Human conversation is essential for sharing nuanced information, asking clarifying questions, and collectively understanding complex situations (sensemaking). This process is hard to replicate with rigid forms.
*   **How You Measure Drives Behavior:** The metrics you choose to track will influence how people work. If you only measure checklist completion, people will optimize for that, potentially at the expense of other critical activities.
*   **Redundancy Can Be a Safety Net:** Having multiple, sometimes overlapping, ways for important information to be communicated can add robustness and safety to a system. Removing redundant channels (like informal conversation) can be risky.

By understanding these principles, you can design better systems, anticipate potential problems, and ensure that your interventions truly lead to better outcomes, rather than unintentionally making things worse.


MULTI-LLM REASONING WORKFLOW COMPLETED SUCCESSFULLY!

OpenAI  : Question generation completed.
Claude  : Solution generation completed.
Gemini  : Detailed explanation completed.

All three AI models have completed their tasks!


In [3]:
# =====================================================
# MULTI-LLM WORKFLOW: OPENAI + CLAUDE + GEMINI
# =====================================================

# STEP 1: Import libraries

import os
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from IPython.display import Markdown, display


# STEP 2: Load environment variables

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "").strip()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "").strip()
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "").strip()
ANTHROPIC_BASE_URL = os.getenv(
    "ANTHROPIC_BASE_URL", "https://api.anthropic.com"
).strip().rstrip("/")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "").strip()


# STEP 3: Validate configuration

if not all([
    OPENAI_API_KEY, OPENAI_MODEL,
    ANTHROPIC_API_KEY, CLAUDE_MODEL,
    GEMINI_API_KEY, GEMINI_MODEL
]):
    raise ValueError("Check API keys and model IDs in your .env file.")



# STEP 4: Initialize AI clients

openai_client = OpenAI(api_key=OPENAI_API_KEY)

claude_client = Anthropic(
    api_key=ANTHROPIC_API_KEY,
    base_url=ANTHROPIC_BASE_URL
)

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("All AI clients initialized successfully!")


# STEP 5: Generate a reasoning question using OpenAI

print("\nGenerating question using OpenAI...")

question_response = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {
            "role": "system",
            "content": """
            You are an expert reasoning assessment designer.
            Generate exactly ONE challenging, non-mathematical
            reasoning question.
            The question must have a clear and defensible answer.
            Return only the question. Do not provide its answer.
            """
        },
        {
            "role": "user",
            "content": "Generate a challenging reasoning question."
        }
    ]
)

question = question_response.choices[0].message.content

display(Markdown("## 🧠 OpenAI - Generated Question\n\n" + question))


# STEP 6: Solve the question using Claude

print("\nSolving question using Claude...")

claude_response = claude_client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=3000,
    system="""
    You are an expert analytical reasoning specialist.
    Solve the given question step by step.
    Explain important factors, consider alternative explanations,
    and provide a clear, logically justified final answer.
    """,
    messages=[
        {"role": "user", "content": question}
    ]
)

claude_solution = "\n".join(
    block.text for block in claude_response.content
    if block.type == "text"
)

display(Markdown("## 🤖 Claude - Solution\n\n" + claude_solution))


# STEP 7: Explain Claude's solution using Gemini
print("\nExplaining solution using Gemini...\n")

prompt = f"""
You are an expert teacher.

Explain the following question and Claude's solution
to students in simple language.

Include:
1. Understanding the question.
2. Step-by-step reasoning.
3. Important concepts.
4. A practical example.
5. Key takeaways.

Question:
{question}

Claude's Solution:
{claude_solution}
"""

gemini_response = gemini_client.models.generate_content_stream(
    model=GEMINI_MODEL,
    contents=prompt
)

gemini_explanation = ""

for chunk in gemini_response:
    if chunk.text:
        print(chunk.text, end="", flush=True)
        gemini_explanation += chunk.text

display(Markdown("## 📚 Gemini - Detailed Explanation\n\n" + gemini_explanation))

print("\n\n✅ MULTI-LLM WORKFLOW COMPLETED SUCCESSFULLY!")

All AI clients initialized successfully!

Generating question using OpenAI...


## 🧠 OpenAI - Generated Question

Five curators—Ava, Ben, Cleo, Dima, and Erin—were questioned after a rare manuscript disappeared. Exactly one of them took it. Each made two claims:

- Ava: “Ben took it.” “Cleo did not take it.”
- Ben: “Dima did not take it.” “I did not take it.”
- Cleo: “Erin took it.” “Dima did not take it.”
- Dima: “Cleo did not take it.” “Ava did not take it.”
- Erin: “I did not take it.” “Dima took it.”

Investigators know that exactly one curator made two true claims, exactly one curator made two false claims, and each of the remaining three made one true claim and one false claim.

Who took the manuscript?


Solving question using Claude...


## 🤖 Claude - Solution

# Answer: **Cleo** took the manuscript.

## Method

Exactly one person is the thief, so I test each of the five possibilities. For each one, I count how many true claims each curator made. The required pattern is:
- exactly one curator with 2 true claims,
- exactly one curator with 0 true claims,
- three curators with 1 true claim each.

## Testing each candidate

**If Ava took it:**
- Ava: F, T → 1
- Ben: T, T → 2
- Cleo: F, T → 1
- Dima: T, F → 1
- Erin: T, F → 1

No one has 0 true claims. ❌

**If Ben took it:**
- Ava: T, T → 2
- Dima: T, T → 2

Two curators have 2 true claims. ❌

**If Cleo took it:**
- Ava ("Ben took it" F; "Cleo did not" F) → **0**
- Ben ("Dima did not" T; "I did not" T) → **2**
- Cleo ("Erin took it" F; "Dima did not" T) → 1
- Dima ("Cleo did not" F; "Ava did not" T) → 1
- Erin ("I did not" T; "Dima took it" F) → 1

This gives one curator with 2 true, one with 0 true, and three with 1 true. ✅

**If Dima took it:**
- Dima: T, T → 2
- Erin: T, T → 2

Two curators have 2 true claims. ❌

**If Erin took it:**
- Ben: T, T → 2
- Cleo: T, T → 2

Two curators have 2 true claims. ❌

## Conclusion

Only the scenario where **Cleo** is the thief fits the investigators' constraints. In that scenario:
- Ben is the curator who told two truths.
- Ava is the curator who told two falsehoods.
- Cleo, Dima, and Erin each made one true and one false claim.


Explaining solution using Gemini...

Hello class! Today, we are going to break down a classic logic puzzle about a missing manuscript. 

Logic puzzles can look intimidating at first, but once you learn how to break them down, they become fun detective games! Let's go through the problem and Claude’s solution step-by-step.

---

### 1. Understanding the Question

Imagine you are a detective investigating a crime. Here are the facts of the case:

* **The Crime:** A rare manuscript was stolen.
* **The Suspects:** 5 curators (Ava, Ben, Cleo, Dima, and Erin).
* **The Rule of Guilt:** **Exactly ONE person is the thief.** (The other 4 are innocent).

Each suspect makes **two statements**. 

The detective knows a secret pattern about their honesty:
1. **1 person** told **2 truths** (True / True)
2. **1 person** told **2 lies** (False / False)
3. **3 people** told **1 truth and 1 lie** (True / False)

**Our Goal:** Figure out who took the manuscript by finding the scenario that matches this ex

## 📚 Gemini - Detailed Explanation

Hello class! Today, we are going to break down a classic logic puzzle about a missing manuscript. 

Logic puzzles can look intimidating at first, but once you learn how to break them down, they become fun detective games! Let's go through the problem and Claude’s solution step-by-step.

---

### 1. Understanding the Question

Imagine you are a detective investigating a crime. Here are the facts of the case:

* **The Crime:** A rare manuscript was stolen.
* **The Suspects:** 5 curators (Ava, Ben, Cleo, Dima, and Erin).
* **The Rule of Guilt:** **Exactly ONE person is the thief.** (The other 4 are innocent).

Each suspect makes **two statements**. 

The detective knows a secret pattern about their honesty:
1. **1 person** told **2 truths** (True / True)
2. **1 person** told **2 lies** (False / False)
3. **3 people** told **1 truth and 1 lie** (True / False)

**Our Goal:** Figure out who took the manuscript by finding the scenario that matches this exact honesty pattern.

---

### 2. Step-by-Step Reasoning (Explaining Claude's Solution)

Since there are only 5 suspects, the smartest strategy is to test each suspect **one by one**. We will pretend Suspect A is guilty, check everyone's statements, and see if it fits our honesty pattern. If it doesn't, we move on to the next suspect!

Let's test them:

#### Scenario 1: Assume AVA took it
* **Ava:** "Ben took it" (False), "Cleo didn't" (True) $\rightarrow$ **1 Truth**
* **Ben:** "Dima didn't" (True), "I didn't" (True) $\rightarrow$ **2 Truths**
* **Cleo:** "Erin took it" (False), "Dima didn't" (True) $\rightarrow$ **1 Truth**
* **Dima:** "Cleo didn't" (True), "Ava didn't" (False) $\rightarrow$ **1 Truth**
* **Erin:** "I didn't" (True), "Dima took it" (False) $\rightarrow$ **1 Truth**
* **Result:** We have four people with 1 truth, and one person with 2 truths. **No one told 2 lies!** ❌ *(Doesn't match our pattern)*

#### Scenario 2: Assume BEN took it
* **Ava:** "Ben took it" (True), "Cleo didn't" (True) $\rightarrow$ **2 Truths**
* **Ben:** "Dima didn't" (True), "I didn't" (False) $\rightarrow$ **1 Truth**
* ...wait! If we check Dima, Dima also gets **2 Truths**. 
* **Result:** We have *two* people with 2 truths. We are only allowed *one*! ❌ *(Doesn't match)*

#### Scenario 3: Assume CLEO took it
Let's test this carefully:
* **Ava:** "Ben took it" (**False**), "Cleo did not take it" (**False**) $\rightarrow$ **0 Truths (2 Lies)**
* **Ben:** "Dima did not take it" (**True**), "I did not take it" (**True**) $\rightarrow$ **2 Truths**
* **Cleo:** "Erin took it" (**False**), "Dima did not take it" (**True**) $\rightarrow$ **1 Truth**
* **Dima:** "Cleo did not take it" (**False**), "Ava did not take it" (**True**) $\rightarrow$ **1 Truth**
* **Erin:** "I did not take it" (**True**), "Dima took it" (**False**) $\rightarrow$ **1 Truth**

Let's count the total honesty pattern for Cleo:
* **2 Lies:** 1 person (Ava)
* **2 Truths:** 1 person (Ben)
* **1 Truth / 1 Lie:** 3 people (Cleo, Dima, Erin)

**Result:** This matches the investigators' clues perfectly! ✅ 

*(Note: If you test Dima or Erin as the thief, you also end up with too many double-truthtellers, so those fail too!)*

---

### 3. Important Concepts

Here are the key logical tools used to solve this:

1. **Proof by Exhaustion (Case-by-Case Testing):** When there are only a few possible answers (only 5 suspects), the easiest method is to test every single possibility until you find the one that works.
2. **True/False Evaluation:** A statement is evaluated relative to an assumption. For example, the statement "Cleo did not take it" is **False** if we assume Cleo *did* take it.
3. **Constraints:** These are the rules given in the problem (e.g., "Exactly one person told 2 truths"). A valid solution must satisfy *every single constraint*.

---

### 4. A Practical Example

Let's simplify this concept with an everyday example:

> **The Cookie Jar Mystery**
> A cookie is missing. Your two brothers, Sam and Tim, each make one claim:
> * Sam: "Tim ate it."
> * Tim: "I didn't eat it."
> 
> *Clue:* One person is lying, and one person is telling the truth. Who ate the cookie?

* **Test 1: Assume Sam ate it.**
  * Sam's claim ("Tim ate it") = False. 
  * Tim's claim ("I didn't eat it") = True. 
  * *Result:* 1 Liar, 1 Truthteller. **This works! Sam ate the cookie.**

* **Test 2: Assume Tim ate it.**
  * Sam's claim ("Tim ate it") = True.
  * Tim's claim ("I didn't eat it") = False.
  * *Result:* 1 Truthteller, 1 Liar. **This also works mathematically!** 

In our original manuscript problem, only *one* candidate creates a clean match with all rules, which leads us uniquely to **Cleo**.

---

### 5. Key Takeaways

1. **Don't Guess—Test:** Instead of trying to guess the thief right away, systematically test each suspect.
2. **Organize Your Data:** Claude used a simple scoring system ($\rightarrow 0, 1, \text{or } 2$ truths per person). Keeping score clearly prevents silly mistakes.
3. **Look for Rule Breakers:** As soon as a scenario violates a rule (like having two double-truthtellers), cross it out immediately and move to the next one!



✅ MULTI-LLM WORKFLOW COMPLETED SUCCESSFULLY!


In [2]:
for model in gemini_client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l